# Granularity
CodeGraphene supports parsing code at different levels of detail: `LINE`, `METHOD`, and `FILE`.
We will point our pipeline at `sample_code.py` and see how changing the granularity alters the graph.

## Setup and Imports

In [1]:
from codegraphene.core import NodeGranularity
from codegraphene.parsers.joern import JoernParser
from codegraphene.trimmers.khop import KHopTrimmer
from codegraphene.serializers.text import CodeReconstructionSerializer
from codegraphene.pipeline import GraphPipeline

target_file = "sample_code.py"

## Line-Level Granularity
Each node represents a single line. We target the query execution (line 63) and look 1 hop away.

In [2]:
pipeline_line = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.LINE),
    trimmer=KHopTrimmer(hops=1),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.LINE)
)

line_output = pipeline_line.run(target_file, target=63)
print("\n--- FINAL LINE PROMPT ---")
print(line_output)

[Pipeline] Parsing sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpx_i6__wa/cpg.bin
[JoernParser] Running: joern-export /tmp/tmpx_i6__wa/cpg.bin --repr all --out /tmp/tmpx_i6__wa/export
[JoernParser] Ingesting DOT file into NetworkX...
[Pipeline] Trimming graph around 63 (Node 25769803796)...
[Pipeline] Trimmed from 2035 to 7 nodes.
[Pipeline] Serializing subgraph...

--- FINAL LINE PROMPT ---
Line 62: tmp7 = self.scheduler
self.scheduler.step()
Line 63: tmp7 = self.scheduler
self.scheduler.step()
Line 66: tmp9


## Method-Level Granularity
All AST tokens inside a function are collapsed into a single "Method" node.
Instead of targeting a line number, we can now target the method by its name!

In [4]:
# With METHOD granularity, the target node must be a string (method name), unlike the line number we passed for LINE granularity.
target_method_name = "step"

pipeline_method = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.METHOD),
    trimmer=KHopTrimmer(hops=2),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.METHOD)
)

method_output = pipeline_method.run(target_file, target=target_method_name)
print("\n--- FINAL METHOD PROMPT ---")
print(method_output)

[Pipeline] Parsing sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpk_ddje2t/cpg.bin
[JoernParser] Running: joern-export /tmp/tmpk_ddje2t/cpg.bin --repr all --out /tmp/tmpk_ddje2t/export
[JoernParser] Ingesting DOT file into NetworkX...
[Pipeline] Trimming graph around 'step' (Node 107374182403)...
[Pipeline] Trimmed from 171 to 56 nodes.
[Pipeline] Serializing subgraph...

--- FINAL METHOD PROMPT ---
:<module>.get_opt_param_group_for_param
:<module>.get_opt_state_for_param
:<module>.epoch_callback_exec
:<module>.load_state_dict
:<module>.gradient_accumulation_boundary
:<module>.clip_grad
:<module>.set_grads
:<module>.load_state_dict
:<module>.get_loss
:<module>.step
:<module>.configure_device
:<module>.is_implemented
:<module>._is_default_fp16
:<module>.recover_states
:<module>.configure_roll_back
:<module>.synchronize_params
:<module>.get_opt_param_group_for_param
:<module>.cache_states
:<module>